# 02 — Feature Engineering

**Objetivo:** Transformar el dataset limpio en un dataset listo para modelado, aplicando las decisiones tomadas en el EDA.

**Transformaciones a aplicar:**
1. Eliminar features sin valor predictivo (`anio_creacion`, `subtipo_interes`)
2. Crear nuevas features derivadas (`es_fin_de_semana`, `franja_horaria`)
3. Agrupar categorías de baja frecuencia en "otros"
4. Encoding de variables categóricas (one-hot y target encoding)
5. Eliminar feature con data leakage (`plataforma_MX_LEAD_QUALIF`)
6. Split train/test estratificado
7. Exportar datasets finales

## 1. Cargar dataset limpio

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

CLEAN_PATH = "../data/processed/leads_cleaned.csv"
df = pd.read_csv(CLEAN_PATH)

print(f"Dataset cargado: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")

## 2. Eliminar features sin valor predictivo

Según el EDA:
- **`anio_creacion`**: Solo 2 valores (2025/2026) con poca variabilidad. No aporta información útil.
- **`subtipo_interes`**: El 96.5% es "Solicitud de compra". Una feature con un solo valor dominante no ayuda al modelo a discriminar.

In [ ]:
cols_eliminar = ["anio_creacion", "subtipo_interes"]

print("Features eliminadas:")
for col in cols_eliminar:
    print(f"  - {col}: {df[col].nunique()} valores únicos")
    print(f"    Distribución: {df[col].value_counts().head(3).to_dict()}")

df = df.drop(columns=cols_eliminar)
print(f"\nDataset: {df.shape[0]:,} filas x {df.shape[1]} columnas")

## 3. Crear nuevas features derivadas

Basado en los hallazgos del EDA:
- **`es_fin_de_semana`**: Binaria (1 = sábado/domingo, 0 = lun-vie). El EDA mostró que los leads de fin de semana convierten más.
- **`franja_horaria`**: Agrupa las 24 horas en 4 franjas. El EDA mostró que la madrugada convierte significativamente más.

In [ ]:
df["es_fin_de_semana"] = df["dia_semana_creacion"].isin(["sábado", "domingo"]).astype(int)

def clasificar_franja(hora):
    if 0 <= hora < 6:
        return "madrugada"
    elif 6 <= hora < 12:
        return "manana"
    elif 12 <= hora < 18:
        return "tarde"
    else:
        return "noche"

df["franja_horaria"] = df["hora_creacion"].apply(clasificar_franja)

print("Nuevas features creadas:\n")
print("es_fin_de_semana:")
print(df["es_fin_de_semana"].value_counts().to_string())
print(f"\nfranja_horaria:")
print(df["franja_horaria"].value_counts().to_string())

print(f"\nConversión por franja horaria:")
for franja in ["madrugada", "manana", "tarde", "noche"]:
    subset = df[df["franja_horaria"]==franja]
    print(f"  {franja:12s} → {subset['target'].mean()*100:.1f}% Hot ({len(subset):,} leads)")

## 4. Agrupar categorías de baja frecuencia

Las categorías con menos del 1% del total se agrupan en "otros" para reducir ruido y dimensionalidad.

In [ ]:
UMBRAL_PCT = 1.0
umbral_abs = int(len(df) * UMBRAL_PCT / 100)

cat_cols_agrupar = ["nombre_formulario", "vehiculo_interes", "origen", "campana", "concesionario"]

print(f"Umbral: {UMBRAL_PCT}% = {umbral_abs} leads\n")

for col in cat_cols_agrupar:
    vc = df[col].value_counts()
    bajas = vc[vc < umbral_abs].index
    if len(bajas) > 0:
        n_antes = df[col].nunique()
        df.loc[df[col].isin(bajas), col] = "otros"
        n_despues = df[col].nunique()
        print(f"  {col}: {n_antes} → {n_despues} categorías ({len(bajas)} agrupadas en 'otros')")
    else:
        print(f"  {col}: sin cambios ({df[col].nunique()} categorías, todas >= umbral)")

## 5. Encoding de variables categóricas

Estrategia de encoding según cardinalidad:
- **One-hot encoding**: `plataforma`, `origen_creacion`, `dia_semana_creacion`, `franja_horaria`, `origen`, `nombre_formulario`, `vehiculo_interes`, `campana`
- **Target encoding**: `concesionario` (aún después de agrupar tiene alta cardinalidad)

**Target encoding** se calcula SOLO con datos de train para evitar data leakage.

### 5.1 Split train/test (antes del encoding)

Se divide el dataset **antes** de aplicar el encoding para evitar **data leakage**. Si calculáramos el target encoding con todo el dataset, el modelo vería información del test durante el entrenamiento, inflando artificialmente las métricas.

- **80% train / 20% test** con `stratify=y` para mantener la proporción de Hot/Cold en ambos conjuntos.

In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]:,} filas ({y_train.mean()*100:.1f}% Hot)")
print(f"Test:  {X_test.shape[0]:,} filas ({y_test.mean()*100:.1f}% Hot)")
print(f"\nStratificación verificada: proporciones similares en train y test")

### 5.2 Target encoding para `concesionario`

`concesionario` tiene 42 categorías (después de agrupar las de baja frecuencia). Aplicar one-hot crearía 41 columnas sparse. En su lugar, se reemplaza cada concesionario por su **tasa de conversión promedio** (% de Hot Leads).

- Se calcula **solo con datos de train** para evitar data leakage.
- Los concesionarios que aparezcan en test pero no en train reciben la **media global** como valor por defecto.
- Resultado: una sola columna numérica `concesionario_target_enc` (float entre 0 y 1).

In [ ]:
target_enc_col = "concesionario"

train_temp = X_train.copy()
train_temp["target"] = y_train.values
concesionario_means = train_temp.groupby(target_enc_col)["target"].mean()
global_mean = y_train.mean()

X_train["concesionario_target_enc"] = X_train[target_enc_col].map(concesionario_means).fillna(global_mean)
X_test["concesionario_target_enc"] = X_test[target_enc_col].map(concesionario_means).fillna(global_mean)

X_train = X_train.drop(columns=[target_enc_col])
X_test = X_test.drop(columns=[target_enc_col])

print(f"Target encoding aplicado a '{target_enc_col}':")
print(f"  Valores únicos mapeados: {len(concesionario_means)}")
print(f"  Rango: {concesionario_means.min():.3f} - {concesionario_means.max():.3f}")
print(f"  Media global (fallback): {global_mean:.3f}")

### 5.3 One-hot encoding para las demás categóricas

Las features categóricas restantes (8 columnas) son **nominales** (no tienen orden natural), por lo que se convierten en columnas binarias (0/1) con one-hot encoding.

- **`drop_first=True`**: Elimina una categoría por feature para evitar multicolinealidad perfecta. La categoría eliminada queda implícita (cuando todas las demás son 0).
- **`reindex`**: Asegura que X_test tenga exactamente las mismas columnas que X_train. Si una categoría aparece en test pero no en train, se ignora; si falta, se llena con 0.

In [ ]:
onehot_cols = [col for col in X_train.select_dtypes(include='object').columns]

print(f"Columnas para one-hot encoding ({len(onehot_cols)}):")
for col in onehot_cols:
    print(f"  - {col}: {X_train[col].nunique()} categorías")

X_train = pd.get_dummies(X_train, columns=onehot_cols, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=onehot_cols, drop_first=True, dtype=int)

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(f"\nDimensiones finales:")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")

### 5.4 Eliminar feature con data leakage: `plataforma_MX_LEAD_QUALIF`

> **Nota de proceso:** Este paso NO se identificó durante el feature engineering original. Se descubrió *después* de completar el modelado (`03_modeling`) y la evaluación (`04_evaluation`), cuando el análisis de importancia de features y SHAP revelaron que `plataforma_MX_LEAD_QUALIF` concentraba el **41.2% de la importancia** del modelo — una señal desproporcionada que levantó sospechas. Al investigar el significado de negocio de esta variable, se confirmó que era la cola donde se asignan los leads *ya clasificados* como calientes. Se documenta aquí (en el paso de feature engineering) para mantener el pipeline limpio y reproducible, pero es importante reconocer que fue un hallazgo tardío que obligó a re-ejecutar todo el pipeline.

**¿Qué es `plataforma_MX_LEAD_QUALIF`?**

Es la cola de CRM donde se asignan los leads que ya fueron clasificados como Hot. Cuando se descargó la base de datos, todos los Hot Leads ya tenían este valor asignado, lo que significa que esta feature **contiene la respuesta que el modelo intenta predecir**.

**¿Por qué es data leakage?**

En producción, cuando llegue un lead nuevo, aún no habrá sido asignado a ninguna cola — eso es justamente lo que el modelo debe predecir. Mantener esta feature inflaría artificialmente las métricas porque el modelo estaría "leyendo la respuesta" en lugar de predecirla.

**¿Por qué se elimina toda la columna `plataforma`?**

`plataforma` solo tenía 2 valores. Con `drop_first=True`, quedó una sola columna: `plataforma_MX_LEAD_QUALIF`. El valor restante (eliminado por `drop_first`) es simplemente el inverso — saber que un lead NO está en MX_LEAD_QUALIF es lo mismo que saber que SÍ está. Por lo tanto, cualquier columna derivada de `plataforma` filtra la misma información.

**Impacto de la corrección:** Al eliminar esta feature y reentrenar, el ROC-AUC bajó de 0.9494 a 0.9476 (caída mínima de 0.2%), confirmando que las demás features ya capturaban la señal real. Las métricas actuales reflejan el rendimiento verdadero en producción.

In [ ]:
leakage_col = "plataforma_MX_LEAD_QUALIF"

if leakage_col in X_train.columns:
    X_train = X_train.drop(columns=[leakage_col])
    X_test = X_test.drop(columns=[leakage_col])
    print(f"Columna '{leakage_col}' eliminada por data leakage.")
    print(f"  X_train: {X_train.shape}")
    print(f"  X_test:  {X_test.shape}")
else:
    print(f"Columna '{leakage_col}' no encontrada (ya fue eliminada).")

### 5.4 Eliminar feature con data leakage: `plataforma_MX_LEAD_QUALIF`

La columna `plataforma_MX_LEAD_QUALIF` indica si el lead fue asignado a la cola de leads calificados (hot). Esto significa que **contiene la respuesta que el modelo intenta predecir**: todos los Hot Leads tienen esta feature en 1 porque fueron movidos a esa cola *después* de ser clasificados.

En producción, cuando llegue un lead nuevo, aún no habrá sido asignado a ninguna cola, por lo que esta feature no estaría disponible. Mantenerla inflaría artificialmente las métricas del modelo (era responsable del **41.2% de la importancia**).

Como `plataforma` solo tenía 2 valores y `drop_first=True` ya eliminó uno, `plataforma_MX_LEAD_QUALIF` es la única columna resultante. Eliminarla remueve toda la información de plataforma, lo cual es correcto porque el valor restante es simplemente el inverso (misma información).

In [ ]:
leakage_col = "plataforma_MX_LEAD_QUALIF"

if leakage_col in X_train.columns:
    X_train = X_train.drop(columns=[leakage_col])
    X_test = X_test.drop(columns=[leakage_col])
    print(f"Columna '{leakage_col}' eliminada por data leakage.")
    print(f"  X_train: {X_train.shape}")
    print(f"  X_test:  {X_test.shape}")
else:
    print(f"Columna '{leakage_col}' no encontrada (ya fue eliminada).")

## 6. Verificación final

Antes de exportar, se valida que el dataset esté listo para modelado. Se verifican 4 puntos críticos:

1. **Dimensiones**: Que train y test tengan el mismo número de features y filas coherentes con el split 80/20.
2. **Nulos**: Que no quede ningún valor nulo (los modelos de scikit-learn no aceptan NaN).
3. **Tipos de datos**: Que todas las columnas sean numéricas (int o float). No deben quedar columnas de tipo `object` (texto).
4. **Lista de features**: Inventario completo de las 49 features para referencia y trazabilidad.

In [ ]:
print("=== VERIFICACIÓN FINAL ===\n")
print(f"X_train: {X_train.shape[0]:,} filas x {X_train.shape[1]} features")
print(f"X_test:  {X_test.shape[0]:,} filas x {X_test.shape[1]} features")
print(f"y_train: {len(y_train):,} ({y_train.mean()*100:.1f}% Hot)")
print(f"y_test:  {len(y_test):,} ({y_test.mean()*100:.1f}% Hot)")

print(f"\nNulos en X_train: {X_train.isnull().sum().sum()}")
print(f"Nulos en X_test:  {X_test.isnull().sum().sum()}")

print(f"\nTipos de datos:")
print(X_train.dtypes.value_counts().to_string())

print(f"\nFeatures finales ({X_train.shape[1]}):")
for col in X_train.columns:
    print(f"  - {col}")

## 7. Exportar datasets para modelado

In [ ]:
import os

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

X_train.to_csv(f"{OUTPUT_DIR}/X_train.csv", index=False)
X_test.to_csv(f"{OUTPUT_DIR}/X_test.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)

print("Archivos exportados:")
for f in ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv"]:
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}")
    print(f"  {f}: {size/1024:.1f} KB")

---

**Feature engineering completado.** Los datasets están listos para el modelado en `03_modeling.ipynb`.

**Resumen de transformaciones:**
- Eliminadas: `anio_creacion`, `subtipo_interes`
- Creadas: `es_fin_de_semana`, `franja_horaria`
- Target encoding: `concesionario` → `concesionario_target_enc`
- One-hot encoding: todas las demás categóricas
- Categorías de baja frecuencia agrupadas en "otros"
- Split 80/20 estratificado por target